## now we build features for all three of the splits

In [20]:
import os
import pandas as pd
import numpy as np
from datetime import date

os.chdir('/Users/anasantana/Desktop/copa')

results   = pd.read_csv('results.csv',      parse_dates=['date'])
shootouts = pd.read_csv('shootouts.csv',    parse_dates=['date'])
fnames    = pd.read_csv('former_names.csv', parse_dates=['start_date', 'end_date'])

name_map = dict(zip(fnames['former'], fnames['current']))
def normalize_team(name): return name_map.get(name, name)
for col in ['home_team','away_team']:
    results[col]   = results[col].map(normalize_team)
    shootouts[col] = shootouts[col].map(normalize_team)
shootouts['winner'] = shootouts['winner'].map(normalize_team)

shootouts_slim = shootouts[['date','home_team','away_team','winner']].rename(columns={'winner':'shootout_winner'})
df = results.merge(shootouts_slim, on=['date','home_team','away_team'], how='left')

def true_winner(row):
    if pd.notna(row['shootout_winner']): return row['shootout_winner']
    elif row['home_score'] > row['away_score']: return row['home_team']
    elif row['away_score'] > row['home_score']: return row['away_team']
    else: return 'Draw'
df['winner'] = df.apply(true_winner, axis=1)

wc2026_matches = df[(df['tournament']=='FIFA World Cup') & (df['date'].dt.year==2026)]
WC2026_TEAMS   = set(wc2026_matches['home_team'].tolist() + wc2026_matches['away_team'].tolist())
both_wc   = df['home_team'].isin(WC2026_TEAMS) & df['away_team'].isin(WC2026_TEAMS)
is_wc2026 = (df['tournament']=='FIFA World Cup') & (df['date'].dt.year==2026)

train           = df[both_wc & (df['date'] < '2023-01-01') & ~is_wc2026].copy().sort_values('date').reset_index(drop=True)
historical_test = df[both_wc & (df['date'] >= '2023-01-01') & ~is_wc2026].copy().sort_values('date').reset_index(drop=True)
wc_test         = df[is_wc2026].copy().sort_values('date').reset_index(drop=True)

TODAY = pd.Timestamp(date.today())
def recency_weight(d):
    days = (TODAY - d).days
    if days <= 365*3: return 3.0
    elif days <= 365*8: return 2.0
    else: return 1.0
MAJOR = {'FIFA World Cup','Copa América','UEFA Euro','Africa Cup of Nations','AFC Asian Cup','CONCACAF Gold Cup'}
def t_weight(t):
    if t in MAJOR: return 2.0
    elif 'qualifier' in t.lower() or 'qualification' in t.lower(): return 1.5
    else: return 1.0
train['recency_weight']    = train['date'].apply(recency_weight)
train['tournament_weight'] = train['tournament'].apply(t_weight)
train['weight']            = train['recency_weight'] * train['tournament_weight']

print(f'train: {len(train):,} | historical_test: {len(historical_test):,} | wc_test: {len(wc_test):,}')

train: 7,124 | historical_test: 403 | wc_test: 72


## 1. Compute team stats from training data only

- **goal_diff** — weighted average (goals scored − goals conceded) per game. A single number that captures net dominance better than attack and defense separately.
- **draw_rate** — how often this team draws. Teams with high draw rates will feed the two-stage model.

In [21]:
def compute_team_stats(train_df):
    home = train_df[['home_team','home_score','away_score','winner','weight']].copy()
    home.columns = ['team','scored','conceded','winner','weight']
    home['won']      = (home['winner'] == home['team']).astype(float)
    home['drew']     = (home['winner'] == 'Draw').astype(float)
    home['goal_diff'] = home['scored'] - home['conceded']

    away = train_df[['away_team','away_score','home_score','winner','weight']].copy()
    away.columns = ['team','scored','conceded','winner','weight']
    away['won']      = (away['winner'] == away['team']).astype(float)
    away['drew']     = (away['winner'] == 'Draw').astype(float)
    away['goal_diff'] = away['scored'] - away['conceded']

    all_games = pd.concat([home, away], ignore_index=True)
    def wavg(g, col): return np.average(g[col], weights=g['weight'])

    return all_games.groupby('team').apply(
        lambda g: pd.Series({
            'attack':    wavg(g, 'scored'),
            'defense':   wavg(g, 'conceded'),
            'win_rate':  wavg(g, 'won'),
            'draw_rate': wavg(g, 'drew'),
            'goal_diff': wavg(g, 'goal_diff'),
            'games':     len(g)
        })
    ).reset_index()

def compute_recent_form(train_df, n=5):
    home = train_df[['date','home_team','winner']].rename(columns={'home_team':'team'})
    away = train_df[['date','away_team','winner']].rename(columns={'away_team':'team'})
    all_games = pd.concat([home, away]).sort_values('date')
    def pts(row):
        if row['winner'] == row['team']: return 3
        elif row['winner'] == 'Draw':    return 1
        else:                            return 0
    all_games['points'] = all_games.apply(pts, axis=1)
    return (all_games.groupby('team').tail(n)
            .groupby('team')['points'].sum().reset_index()
            .rename(columns={'points':'form'}))

def h2h_win_rate(team_a, team_b, train_df):
    mask = (((train_df['home_team']==team_a)&(train_df['away_team']==team_b))|
            ((train_df['home_team']==team_b)&(train_df['away_team']==team_a)))
    h2h = train_df[mask].copy()
    if len(h2h) == 0: return 0.5
    h2h['a_won'] = (h2h['winner'] == team_a).astype(float)
    return np.average(h2h['a_won'], weights=h2h['weight'])

def h2h_draw_rate(team_a, team_b, train_df):
    """How often these two teams specifically draw against each other."""
    mask = (((train_df['home_team']==team_a)&(train_df['away_team']==team_b))|
            ((train_df['home_team']==team_b)&(train_df['away_team']==team_a)))
    h2h = train_df[mask].copy()
    if len(h2h) == 0: return 0.25  # global draw rate baseline
    h2h['drew'] = (h2h['winner'] == 'Draw').astype(float)
    return np.average(h2h['drew'], weights=h2h['weight'])

def build_stats_lookup(train_df):
    ts = compute_team_stats(train_df)
    rf = compute_recent_form(train_df)
    ts = ts.merge(rf, on='team', how='left')
    return ts.set_index('team').to_dict('index')

stats_lookup = build_stats_lookup(train)
def get_stat(team, stat): return stats_lookup.get(team, {}).get(stat, 0)

print('Team stats computed.')
team_stats = compute_team_stats(train).merge(compute_recent_form(train), on='team')
team_stats.sort_values('goal_diff', ascending=False).head(10)

Team stats computed.


/var/folders/gt/srhy4gnd3js3k460dfgpn0q40000gn/T/ipykernel_23908/3688843165.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return all_games.groupby('team').apply(
/var/folders/gt/srhy4gnd3js3k460dfgpn0q40000gn/T/ipykernel_23908/3688843165.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return all_games.groupby('team').apply(


,team,attack,defense,win_rate,draw_rate,goal_diff,games,form
6,Brazil,1.953362,0.952820,0.592191,0.202278,1.000542,648.0,12
16,England,1.848370,1.119030,0.483700,0.238059,0.729340,514.0,8
1,Argentina,1.793319,1.096555,0.526618,0.220772,0.696764,677.0,15
18,Germany,1.971302,1.304636,0.537896,0.207506,0.666667,518.0,4
40,Spain,1.695567,1.057143,0.501478,0.243350,0.638424,366.0,7
28,Netherlands,1.848558,1.394231,0.434295,0.237179,0.454327,486.0,10
26,Mexico,1.526273,1.256979,0.434319,0.215107,0.269294,461.0,6
17,France,1.665605,1.420382,0.481688,0.200637,0.245223,479.0,9
27,Morocco,1.169096,0.925656,0.402332,0.301749,0.243440,251.0,9
41,Sweden,1.617316,1.441558,0.378355,0.235498,0.175758,458.0,9


## 2. Build feature matrix

- `is_neutral` — whether the match is on neutral ground (WC games always are; home games in training have real home advantage, which is noise for our prediction task)
- `a_goal_diff`, `b_goal_diff`, `goal_diff_diff` — net dominance per game
- `a_draw_rate`, `b_draw_rate` — how draw-prone each team is (feeds the two-stage model)
- `h2h_draw_rate` — how often this specific matchup ends in a draw

In [22]:
def build_features(match_df, train_df, label=True):
    lookup = build_stats_lookup(train_df)
    def gs(team, stat): return lookup.get(team, {}).get(stat, 0)

    rows = []
    for _, row in match_df.iterrows():
        a, b = row['home_team'], row['away_team']
        neutral = 1 if str(row.get('neutral', False)).upper() == 'TRUE' else 0

        feat = {
            'team_a': a, 'team_b': b, 'date': row['date'],
            # Raw stats
            'a_attack':    gs(a,'attack'),    'b_attack':    gs(b,'attack'),
            'a_defense':   gs(a,'defense'),   'b_defense':   gs(b,'defense'),
            'a_win_rate':  gs(a,'win_rate'),  'b_win_rate':  gs(b,'win_rate'),
            'a_draw_rate': gs(a,'draw_rate'), 'b_draw_rate': gs(b,'draw_rate'),
            'a_goal_diff': gs(a,'goal_diff'), 'b_goal_diff': gs(b,'goal_diff'),
            'a_form':      gs(a,'form'),      'b_form':      gs(b,'form'),
            # Difference features
            'attack_diff':    gs(a,'attack')    - gs(b,'attack'),
            'defense_diff':   gs(a,'defense')   - gs(b,'defense'),
            'win_rate_diff':  gs(a,'win_rate')  - gs(b,'win_rate'),
            'goal_diff_diff': gs(a,'goal_diff') - gs(b,'goal_diff'),
            'form_diff':      gs(a,'form')      - gs(b,'form'),
            # Head-to-head
            'h2h_a_win_rate': h2h_win_rate(a, b, train_df),
            'h2h_draw_rate':  h2h_draw_rate(a, b, train_df),
            # Context
            'is_neutral': neutral,
        }
        if label:
            if row['winner'] == a:        feat['outcome'] = 1
            elif row['winner'] == 'Draw': feat['outcome'] = 0
            else:                         feat['outcome'] = -1
        rows.append(feat)
    return pd.DataFrame(rows)

train_features     = build_features(train,           train)
hist_test_features = build_features(historical_test, train)
wc_test_features   = build_features(wc_test,         train)

print(f'train_features:      {train_features.shape}')
print(f'hist_test_features:  {hist_test_features.shape}')
print(f'wc_test_features:    {wc_test_features.shape}')

/var/folders/gt/srhy4gnd3js3k460dfgpn0q40000gn/T/ipykernel_23908/3688843165.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return all_games.groupby('team').apply(
/var/folders/gt/srhy4gnd3js3k460dfgpn0q40000gn/T/ipykernel_23908/3688843165.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return all_games.groupby('team').apply(


train_features:      (7124, 24)
hist_test_features:  (403, 24)
wc_test_features:    (72, 24)


/var/folders/gt/srhy4gnd3js3k460dfgpn0q40000gn/T/ipykernel_23908/3688843165.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return all_games.groupby('team').apply(


In [24]:
label_map = {1: 'A wins', 0: 'Draw', -1: 'B wins'}
for name, feat_df in [('train', train_features), ('historical_test', hist_test_features), ('wc_test', wc_test_features)]:
    counts = feat_df['outcome'].value_counts().sort_index()
    print(f'\n{name}:')
    for k, v in counts.items():
        print(f'  {label_map[k]}: {v} ({v/len(feat_df):.1%})')

#outcome distribution across all splits


train:
  B wins: 2065 (29.0%)
  Draw: 1666 (23.4%)
  A wins: 3393 (47.6%)

historical_test:
  B wins: 134 (33.3%)
  Draw: 97 (24.1%)
  A wins: 172 (42.7%)

wc_test:
  B wins: 9 (12.5%)
  Draw: 38 (52.8%)
  A wins: 25 (34.7%)
